<a href="https://colab.research.google.com/github/mc-castro/clinicaltrials-ia-thesis/blob/maria%2Fdevelop/notebooks/05_ranker_patients_with_xai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Imports

In [ ]:
import google.generativeai as genai
import pandas as pd
import json
import time
from tqdm import tqdm
from google.colab import userdata, drive
import os

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [ ]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Ranker

In [ ]:
class ClinicalRanker:
    def __init__(self, api_key, version_genai):
        genai.configure(api_key=userdata.get(api_key))
        self.model = genai.GenerativeModel(version_genai)
        # Standardized Clinical Weights
        self.weights = {
            "gold": 40,   # Invasive procedure confirmation
            "diag": 30,   # Explicit diagnosis/imaging
            "phys": 20,   # Physical exam findings
            "symp": 10    # Subjective symptoms
        }

    def _process_ranking_batch(self, patients_list, study_criteria):
        prompt = f"""
        TASK: Act as a senior clinical trial investigator. Analyze this ELIGIBLE patient for a clinical trial ranking.
        STUDY CRITERIA: {study_criteria}

        CRITICAL CHECK:
        Check if 'list_procedures' or 'HPI' contains any procedure that is FORBIDDEN by the study's exclusion criteria.

        For each patient, extract these boolean flags (True/False):
        1. GOLD_EVIDENCE: Definitive evidence of all inclusion criteria, no confounding factors.
        2. DIAGNOSIS_EVIDENCE: Explicit diagnosis or definitive imaging mentioned in records.
        3. PHYSICAL_EVIDENCE: Supporting physical exam findings.
        4. SYMPTOM_EVIDENCE: Subjective complaints/symptoms.
        5. EXCLUSION_FOUND: Does the patient have any condition explicitly FORBIDDEN by the study? (e.g., previous procedure, active cancer, specific comorbidity).
        6. ATTRITION_RISK: Is the patient unlikely to complete the study?
           (Criteria: Extreme age > 90, terminal illness/palliative care, severe dementia, or hemodynamic instability).

        PATIENTS DATA (JSON):
        {patients_list}

        RETURN ONLY A JSON LIST OF OBJECTS:
        [
          {{
            "subject_id": int,
            "gold": bool,
            "diag": bool,
            "phys": bool,
            "symp": bool,
            "exclusion": bool,
            "risk": bool,
            "justification": "Short clinical reasoning in English"
          }}
        ]
        """
        try:
            response = self.model.generate_content(prompt, generation_config={"response_mime_type": "application/json"})
            return json.loads(response.text)
        except Exception as e:
            print(f"\n[API ERROR] {e}")
            return []

    def calculate_score(self, res):
        # 1. Hard Exclusion: If any exclusion criteria found, score is 0
        if res.get('exclusion'):
            return 0

        # 2. Base Score Calculation
        score = sum(self.weights[k] for k in self.weights if res.get(k))

        # 3. Attrition Penalty: Reduce 30% if high risk of drop-out/instability
        if res.get('risk'):
            score = int(score * 0.7)

        return score

    def run(self, report_df, patients_df, studies_df, output_file="final_ranking_results.csv"):
        processed_ncts = []
        if os.path.exists(output_file):
            temp_df = pd.read_csv(output_file)
            processed_ncts = temp_df['nct_id'].unique().tolist()

        for _, row in tqdm(report_df.iterrows(), total=len(report_df), desc="Ranking Processes"):
            nct_id = row['nct_id']
            if nct_id in processed_ncts: continue

            eligible_ids = json.loads(row['subject_ids'])
            if not eligible_ids: continue

            study_info = studies_df[studies_df['nct_id'] == nct_id].iloc[0]
            criteria = f"Inclusion: {study_info['inclusion_criteria']}\nExclusion: {study_info['exclusion_criteria']}"

            subset_patients = patients_df[patients_df['subject_id'].isin(eligible_ids)].to_dict(orient='records')

            study_results = []
            # Batch size of 5 for speed and context window safety
            for i in range(0, len(subset_patients), 5):
                batch = subset_patients[i : i + 5]
                clean_batch = [{k: v for k, v in p.items() if pd.notna(v)} for p in batch]

                results = self._process_ranking_batch(json.dumps(clean_batch), criteria)

                for res in results:
                    final_score = self.calculate_score(res)
                    study_results.append({
                        "nct_id": nct_id,
                        "subject_id": res.get('subject_id'),
                        "score": final_score,
                        "gold": res.get('gold'),
                        "diag": res.get('diag'),
                        "phys": res.get('phys'),
                        "symp": res.get('symp'),
                        "exclusion": res.get('exclusion'),
                        "risk": res.get('risk'),
                        "justification": res.get('justification')
                    })
                time.sleep(4)

            if study_results:
                pd.DataFrame(study_results).to_csv(output_file, mode='a', index=False, header=not os.path.exists(output_file))

        # return pd.read_csv(output_file) if os.path.exists(output_file) else pd.DataFrame()

## Main

In [ ]:
# @title Extract Data
report_df = pd.read_csv("/content/drive/My Drive/Mestrado/Dissertação/mimic-iv-ext-cardiac-disease/processed/matching_report.csv")
mimic_patients_df = pd.read_parquet("/content/drive/My Drive/Mestrado/Dissertação/mimic-iv-ext-cardiac-disease/processed/df_diag_final_elegibility.parquet")

clinical_trials_df = pd.read_csv("/content/drive/My Drive/Mestrado/Dissertação/mimic-iv-ext-cardiac-disease/processed/processed_studies.csv")

all_cols = mimic_patients_df.columns.tolist()
cols_obrigatorias = ['subject_id', 'gender', 'age', 'name_diag_principal', 'list_procedures', 'HPI', 'chief_complaint']
to_ignore = ['hadm_id']
cols_com_dados = mimic_patients_df.columns[mimic_patients_df.notna().sum() > 0].tolist()
filtered_cols = [c for c in cols_com_dados if c not in to_ignore and not c.endswith('_FLAG')]
cols_finais = list(set(cols_obrigatorias + filtered_cols))
patients_subset = mimic_patients_df[cols_finais].copy()

In [ ]:
# @title Generate Rank
ranker = ClinicalRanker(
    api_key="RankerGeminiAPI",
    version_genai = 'gemini-2.5-flash')

ranker.run(report_df,
            patients_subset,
            clinical_trials_df,
            output_file="/content/drive/My Drive/Mestrado/Dissertação/mimic-iv-ext-cardiac-disease/processed/ranking_incremental.csv")

Ranking Processes:  30%|███       | 3/10 [1:31:44<3:37:31, 1864.48s/it]


[API ERROR] Expecting ',' delimiter: line 30 column 87 (char 1005)

[API ERROR] Expecting ',' delimiter: line 51 column 3 (char 1816)


Ranking Processes: 100%|██████████| 10/10 [3:12:37<00:00, 1155.73s/it]
